# 05 Advanced Analytics

VaR/CVaR, rolling Sharpe, cohort behavior, SIP continuity, sector concentration, recommender, and optional simulation/optimization extensions.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHARTS_DIR = REPORTS_DIR / "charts"
sns.set_theme(style="whitegrid")
PROJECT_ROOT

In [ ]:
var = pd.read_csv(REPORTS_DIR / "var_cvar_report.csv")
hhi = pd.read_csv(REPORTS_DIR / "sector_hhi.csv")
display(var.sort_values("var_95").head(10))
display(hhi.sort_values("sector_hhi", ascending=False).head(10))

In [ ]:
txn = pd.read_csv(PROCESSED_DIR / "investor_transactions.csv", parse_dates=["transaction_date"])
txn["first_year"] = txn.groupby("investor_id")["transaction_date"].transform("min").dt.year
cohort = txn.groupby("first_year").agg(
    investors=("investor_id", "nunique"),
    avg_amount=("amount_inr", "mean"),
    total_invested=("amount_inr", "sum")
).reset_index()
display(cohort)

In [ ]:
sip = txn[txn["transaction_type"] == "SIP"].sort_values(["investor_id", "transaction_date"]).copy()
sip["gap_days"] = sip.groupby("investor_id")["transaction_date"].diff().dt.days
continuity = sip.groupby("investor_id").agg(
    sip_txns=("transaction_date", "size"),
    avg_gap_days=("gap_days", "mean")
).query("sip_txns >= 6").reset_index()
continuity["at_risk"] = continuity["avg_gap_days"] > 35
display(continuity.head())
print("At-risk SIP continuity rate:", continuity["at_risk"].mean())

In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT / "scripts"))
from recommender import recommend
display(recommend("Moderate", top_n=3))

## Advanced Insight Starters

1. Funds with the most negative VaR deserve closer drawdown communication in advisor workflows.
2. Cohort-level total invested separates investor acquisition volume from ticket-size quality.
3. SIP continuity flags investors whose contribution gap exceeds 35 days for retention follow-up.
4. Sector HHI identifies concentrated funds where single-sector shocks can dominate outcomes.
5. Risk-appetite recommendations should combine scorecard rank with investor suitability, not score alone.